In [1]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.mps.is_available():
    device = torch.device("mps")

In [2]:
import json
with open("data/traces.jsonl", "r") as f:
    traces = [json.loads(line) for line in f.readlines()]
import pandas as pd

df = pd.DataFrame(traces)

In [3]:
from src.constants import TOP_RFHS_BY_LAYER_HEAD

# Transform TOP_RFHS_BY_LAYER_HEAD to map layer -> [rfh_indices] for each model
layer_to_top_rfhs = {}
for k, v in TOP_RFHS_BY_LAYER_HEAD.items():
    d_tmp = {}
    for cord in v:
        if cord[0] not in d_tmp:
            d_tmp[cord[0]] = []
        d_tmp[cord[0]].append(cord[1])
    layer_to_top_rfhs[k] = d_tmp
layer_to_top_rfhs["qwen-1p5B"]

{16: [2, 11, 0], 1: [5], 19: [1, 5], 12: [1], 23: [2], 14: [3], 20: [9]}

In [4]:
from src.patcher import ActivationPatcher
from src.constants import MODELS_LITERAL

model_alias: MODELS_LITERAL = "qwen-1p5B"

# instantiate patcher
patcher = ActivationPatcher(model_alias, device)

/Users/rishidinesh/Projects/causal-mediation-analysis-rfh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!


Loaded pretrained model deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B into HookedTransformer


In [5]:
# filter dataframe and layer_rfh map for the specific model
df = df[df["model_name"] == model_alias].reset_index(drop=True)
layer_to_top_rfhs = layer_to_top_rfhs[model_alias]
layer_to_top_rfhs

{16: [2, 11, 0], 1: [5], 19: [1, 5], 12: [1], 23: [2], 14: [3], 20: [9]}

In [6]:
import os
from pathlib import Path

save_frequency = 1
start = 13
savepath = f"data/output/activation_patching/{model_alias}/all_rfh_heads.jsonl"
os.makedirs(Path(savepath).parent, exist_ok=True)
results = []
for i, row in df.iterrows():
    if i < start:
        continue
    row = row.to_dict()
    res = patcher.run(
        response_withR = row["response_withR"],
        response_withoutR = row["response_withoutR"],
        heads_by_layer = layer_to_top_rfhs
    )
    print(f"{i} | ID: {row['unique_id']} | withR_loss: {res['withR_loss']:.4f} | withoutR_loss: {res['withoutR_loss']:.4f} | patched_withR_loss: {res['patched_withR_loss']:.4f} | patched_withoutR_loss: {res['patched_withoutR_loss']:.4f}")
    res.update({
        "unique_id": row["unique_id"],
        "model": row["model_name"]
    })
    results.append(res)
    if (i + 1) % save_frequency == 0:
        # print(f"Processed {i + 1} examples, saving intermediate results to {savepath}")
        with open(savepath, "a") as f:
            for r in results:
                f.write(json.dumps(r) + "\n")
        results = []
# save any remaining results
if results:
    print(f"Saving final results to {savepath}")
    with open(savepath, "a") as f:
        for r in results:
            f.write(json.dumps(r) + "\n")
results = []

RuntimeError: MPS backend out of memory (MPS allocated: 24.35 GiB, other allocations: 720.00 KiB, max allowed: 27.20 GiB). Tried to allocate 4.08 GiB on private pool. Use PYTORCH_MPS_HIGH_WATERMARK_RATIO=0.0 to disable upper limit for memory allocations (may cause system failure).

In [ ]:
# import os
# from pathlib import Path

# save_frequency = 10

# for layer in layer_to_top_rfhs:
#     print(f"Layer {layer}: RFH indices {layer_to_top_rfhs[layer]}")
#     for rfh_index in layer_to_top_rfhs[layer]:
#         print(f"Patching RFH index {rfh_index}")
#         savepath = f"data/output/activation_patching/{model_alias}/individual_layer_heads.jsonl"
#         os.makedirs(Path(savepath).parent, exist_ok=True)
#         results = []
#         for idx, row in df.iterrows():
#             row = row.to_dict()
#             res = patcher.run(
#                 response_withR = row["response_withR"],
#                 response_withoutR = row["response_withoutR"],
#                 heads_by_layer = {layer: [rfh_index]}
#             )
#             print(f"ID: {row['example_id']} | withR_loss: {res['withR_loss']:.4f} | withoutR_loss: {res['withoutR_loss']:.4f} | patched_withR_loss: {res['patched_withR_loss']:.4f} | patched_withoutR_loss: {res['patched_withoutR_loss']:.4f}")
#             res.update({
#                 "unique_id": row["unique_id"],
#                 "model": row["model_name"],
#                 "layer": layer,
#                 "head": rfh_index
#             })
#             results.append(res)
#             if (idx + 1) % save_frequency == 0:
#                 print(f"Processed {idx + 1} examples, saving intermediate results to {savepath}")
#                 with open(savepath, "a") as f:
#                     for r in results:
#                         f.write(json.dumps(r) + "\n")
#                 results = []
#         # save any remaining results
#         if results:
#             print(f"Saving final results to {savepath}")
#             with open(savepath, "a") as f:
#                 for r in results:
#                     f.write(json.dumps(r) + "\n")

{'model_name': 'qwen-1p5B', 'unique_id': 'test/precalculus/807.json', 'problem': 'Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$', 'response_withR': "<｜begin▁of▁sentence｜>Convert the point $(0,3)$ in rectangular coordinates to polar coordinates.  Enter your answer in the form $(r,\\theta),$ where $r > 0$ and $0 \\le \\theta < 2 \\pi.$ Please reason step by step, and put your final answer within \\boxed{}. <think>\nOkay, so I need to convert the rectangular coordinates (0, 3) to polar coordinates. Hmm, I remember that polar coordinates are represented as (r, θ), where r is the distance from the origin to the point, and θ is the angle measured from the positive x-axis to the point. Let me try to recall the formulas for converting from rectangular to polar coordinates.\n\nI think the formula for r is the square root of (x squared plus y squared). So, if I plug in x = 0 an